# Grammar Scoring Engine for Spoken Data — final model: speech features + frozen embeddings → ridge regression
**Goal.** Predict a 0-5 MOS grammar score from a 45-60 s speech recording (769 train / 216 test). Metrics: RMSE and Pearson correlation.

**Final model (one model, no ensemble).** Whisper + literal wav2vec2-CTC transcripts → grammar / disfluency / syntax features, acoustic features, frozen wav2vec2 and Whisper-encoder audio embeddings and frozen sentence / DeBERTa text embeddings → in-fold scaling + PCA → **RidgeCV**.
Why this one: in speaker-grouped cross-validation (the only validation that mimics the test set, whose speakers/channels are not in train) linear models on the embeddings beat SVR, boosted trees and every stack of them.

**Validation.** 3x repeated stratified 5-fold CV in which recordings of the same speaker/channel cluster (and exact duplicates) never straddle a fold. The required **training-set RMSE** is reported in Section 16.

**Runtime.** With the cached outputs of earlier runs (attach the previous notebook output as an input) the whole notebook takes a few minutes. From scratch on a T4: Whisper large-v3 ~25 min, wav2vec2 ~10 min, Whisper encoder ~10 min, CoLA/LM/GEC ~25 min, text embeddings ~5 min, audio features ~20 min (CPU), model ~2 min.

**Outputs** (in `/kaggle/working`): `submission.csv` (format of `sample_submission.csv`: `filename,label`), plus helper files for the other id list.

## 2. Imports & optional installs

In [ ]:
# ---- Optional first-run installs (needs Kaggle "Internet: on") -------------------------------
# WHY: the heavy NLP/ASR dependencies are not all pre-installed. Run once, then attach the
# resulting /kaggle/working/cache as a Dataset so later notebooks (and offline re-runs) skip all of it.
INSTALL = True
if INSTALL:
    import subprocess, sys
    pkgs = ['faster-whisper', 'sentence-transformers', 'spacy', 'language_tool_python',
            'wordfreq', 'catboost', 'lightgbm', 'xgboost']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=False)
    # LanguageTool needs a JRE:  apt-get install -y default-jre-headless
    # If faster-whisper/ctranslate2 complains about cuDNN, pin ctranslate2==4.4.0.


In [ ]:
import os, re, sys, json, math, time, glob, gc, random, hashlib, warnings, difflib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import nnls
from scipy.signal import find_peaks
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import soundfile as sf
import librosa
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeCV, ElasticNetCV, LogisticRegression, LinearRegression
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.dummy import DummyRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

# Optional heavy libraries: every one of them is guarded so the notebook degrades gracefully.
def _try_import(name):
    try:
        return __import__(name)
    except Exception:
        return None

lgb = _try_import('lightgbm')
xgb = _try_import('xgboost')
cb = _try_import('catboost')
torch = _try_import('torch')
HAS_GPU = bool(torch is not None and torch.cuda.is_available())
DEVICE = 'cuda' if HAS_GPU else 'cpu'

def seed_everything(seed):
    # WHY: reproducibility. (GPU kernels may still be non-deterministic; we only use them for frozen inference.)
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if HAS_GPU: torch.cuda.manual_seed_all(seed)

print(f"GPU: {HAS_GPU} | lightgbm={lgb is not None} xgboost={xgb is not None} catboost={cb is not None} torch={torch is not None}")


## 3. Configuration
Every expensive stage has a switch and a model name. Offline? Put model folders in `CFG.MODEL_OVERRIDES`.

In [ ]:
# ---- Configuration -----------------------------------------------------------------------------
# WHY one config object: every expensive step has a switch + a model name, so you can trade
# accuracy for runtime (or run offline with cached outputs) without touching the pipeline code.
class CFG:
    SEED = 42
    N_SPLITS = 5
    N_REPEATS = 3                    # repeated CV -> 15 fits per model
    SR = 16_000
    LABEL_BOUNDS = (0.0, 5.0)        # rubric range
    N_JOBS = max(1, os.cpu_count() or 1)
    PAUSE_MIN_S = 0.30               # a silent gap >= 300 ms counts as a pause

    # --- ASR (Section 8)
    USE_WHISPER = True
    WHISPER_MODEL = 'large-v3' if HAS_GPU else 'small.en'     # CPU: 'base.en' if you are short on time
    WHISPER_COMPUTE = 'float16' if HAS_GPU else 'int8'
    # Verbatim-leaning prompt: Whisper imitates the prompt's style, so fillers/repetitions survive.
    WHISPER_PROMPT = 'Umm, let me think like, hmm... Okay, here is what I am, like, thinking.'

    # --- literal CTC ASR + SSL audio embeddings (Section 9)
    USE_W2V = True
    W2V_MODEL = 'facebook/wav2vec2-large-960h-lv60-self' if HAS_GPU else 'facebook/wav2vec2-base-960h'

    # --- linguistic models (Section 11)
    USE_SPACY = True;  SPACY_MODEL = 'en_core_web_sm'
    USE_COLA = True;   COLA_MODEL = 'textattack/roberta-base-CoLA'           # grammatical-acceptability classifier
    USE_LM = True;     LM_MODEL = 'gpt2'                                     # token surprisal
    USE_GEC = HAS_GPU; GEC_MODEL = 'vennify/t5-base-grammar-correction'; GEC_PREFIX = 'grammar: '
    USE_LANGUAGETOOL = True

    # --- embeddings (Section 12)
    USE_ST_EMB = True;     ST_MODEL = 'sentence-transformers/all-mpnet-base-v2'
    USE_DEBERTA_EMB = HAS_GPU; DEBERTA_MODEL = 'microsoft/deberta-v3-base'
    EMB_PCA = 32                     # in-fold PCA components per embedding block (default)
    EMB_PCA_BY_BLOCK = {'embw2v': 96, 'embwenc': 96}   # audio blocks profit from more components (PCA sweep)

    # --- validation / runtime settings
    SPK_CLUSTERS = 120               # speaker/channel clusters used as CV groups (more clusters = milder CV; see Section 14 output)
    AUDIO_PARALLEL = 'sequential'    # 'sequential' (safe: librosa.pyin + joblib/loky has segfaulted) | 'threads' | 'loky'


    # --- offline Kaggle: map a HF model id -> local path of an attached model dataset
    MODEL_OVERRIDES = {}

    # --- paths
    WORK = Path(os.environ.get('GSE_WORK_DIR', '/kaggle/working' if Path('/kaggle/working').exists() else './working'))
    CACHE = None

CFG.CACHE = CFG.WORK / 'cache'
CFG.CACHE.mkdir(parents=True, exist_ok=True)

def mpath(name):
    """Resolve a model id to an offline path when one was provided."""
    return CFG.MODEL_OVERRIDES.get(name, name)

def tag(s):
    return re.sub(r'[^A-Za-z0-9]+', '-', str(s)).strip('-')

seed_everything(CFG.SEED)
print('Work dir:', CFG.WORK, '| cache:', CFG.CACHE)

# Re-use caches from a previous run: attach the earlier notebook output (or a Dataset containing its `cache/` folder)
# as an input and the expensive ASR / embedding results are copied in automatically (nothing is recomputed).
import shutil
for _src in glob.glob('/kaggle/input/**/cache', recursive=True):
    _src = Path(_src)
    if _src.is_dir() and _src.resolve() != CFG.CACHE.resolve():
        for _f in _src.glob('*'):
            if _f.is_file() and not (CFG.CACHE / _f.name).exists(): shutil.copy(_f, CFG.CACHE / _f.name)


## 4. Load dataset
Columns are inferred from `sample_submission.csv`; audio is located by walking the data folder (`Dataset_Final/train`, `Dataset_Final/test`).

In [ ]:
# ---- 4. Load dataset -----------------------------------------------------------------------------
FEATURE_BLOCKS = {}   # name -> DataFrame indexed by filename (engineered features)
EMB_BLOCKS = {}       # name -> ndarray aligned with ALL (frozen embeddings)

def find_data_dir():
    env = os.environ.get('GSE_DATA_DIR')
    if env: return Path(env)
    known = Path('/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final')   # layout seen in Data Explorer
    if (known / 'train.csv').exists(): return known
    hits = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True))
    if hits: return Path(hits[0]).parent
    if Path('train.csv').exists(): return Path('.')
    raise FileNotFoundError('train.csv not found - attach the competition data to the notebook.')

DATA_DIR = find_data_dir()
train_raw = pd.read_csv(DATA_DIR / 'train.csv'); test_raw = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print('DATA_DIR:', DATA_DIR)
print('train', train_raw.shape, '| test', test_raw.shape, '| sample_submission', sample_sub.shape)
print('train columns:', list(train_raw.columns), '| test columns:', list(test_raw.columns), '| submission columns:', list(sample_sub.columns))

# Do NOT hard-code column names: infer them from sample_submission / train.
ID_COL = sample_sub.columns[0]
TARGET_COL = [c for c in sample_sub.columns if c != ID_COL][0]
if ID_COL not in train_raw.columns:
    ID_COL = next((c for c in train_raw.columns if 'file' in c.lower()), train_raw.columns[0])
if TARGET_COL not in train_raw.columns:
    TARGET_COL = [c for c in train_raw.columns if c != ID_COL and pd.api.types.is_numeric_dtype(train_raw[c])][0]
print(f'ID_COL={ID_COL!r}  TARGET_COL={TARGET_COL!r}')

def strip_ext(x): return re.sub(r'\.(wav|flac|mp3|ogg|m4a)$', '', str(x).strip(), flags=re.I)
# NOTE: train and test number their files separately, so the SAME name can exist in both folders (different audio!).
# Every recording therefore gets an internal unique key  "<split>/<name>";  raw names are kept for the submission.
tr = train_raw[[ID_COL, TARGET_COL]].rename(columns={ID_COL: 'raw', TARGET_COL: 'y'}).copy(); tr['raw'] = tr.raw.map(strip_ext)
test_ids = [strip_ext(x) for x in test_raw[ID_COL]]; sub_ids = [strip_ext(x) for x in sample_sub[ID_COL]]
print('\nmissing (train):', train_raw.isna().sum().to_dict(), '| missing (test):', test_raw.isna().sum().to_dict())
print('duplicate names inside train:', tr.raw.duplicated().sum(), '| inside test:', len(test_ids) - len(set(test_ids)),
      '| raw-name overlap train/test (expected, different audio):', len(set(tr.raw) & set(test_ids)))
print(f'test.csv ids: {len(set(test_ids))} | sample_submission ids: {len(set(sub_ids))} | in both: {len(set(test_ids) & set(sub_ids))} '
      f'| only in test.csv: {len(set(test_ids) - set(sub_ids))} | only in sample_submission: {len(set(sub_ids) - set(test_ids))}')
# Predict for the UNION so every id the submission needs gets a prediction.
te = pd.DataFrame({'raw': list(dict.fromkeys(test_ids + sub_ids))})

# Index every audio file on disk once, then resolve names (with or without extension) per split.
AUDIO_INDEX = defaultdict(list)
for p in Path(DATA_DIR).rglob('*'):
    if p.suffix.lower() in ('.wav', '.flac', '.mp3', '.ogg', '.m4a'):
        AUDIO_INDEX[p.name].append(p); AUDIO_INDEX[p.stem].append(p)
print('audio files indexed:', len({str(p) for v in AUDIO_INDEX.values() for p in v}))

def resolve_audio(fname, split):
    c = AUDIO_INDEX.get(fname) or AUDIO_INDEX.get(Path(fname).stem) or AUDIO_INDEX.get(fname + '.wav') or []
    pref = [p for p in c if any(split in part.lower() for part in p.parts)]
    if pref: return pref[0]
    # NEVER fall back to the other split's audio (names collide across train/test). Only accept flat layouts without split folders.
    other = 'test' if split == 'train' else 'train'
    flat = [p for p in c if not any(('train' in part.lower()) or ('test' in part.lower()) for part in p.parts[-3:])]
    return flat[0] if flat else None

print('first ids  test.csv:', test_ids[:5], '| sample_submission:', sub_ids[:5], '| train:', list(tr.raw[:3]))
tr['split'] = 'train'; te['split'] = 'test'; te['y'] = np.nan
tr['filename'] = 'train/' + tr.raw; te['filename'] = 'test/' + te.raw
ALL = pd.concat([tr, te], ignore_index=True)
ALL['path'] = [resolve_audio(r, sp) for r, sp in zip(ALL.raw, ALL.split)]
assert ALL.filename.is_unique, 'internal keys must be unique'
miss = ALL[ALL.path.isna()]
print('unresolved audio paths -> train:', int((miss.split == 'train').sum()), '| test:', int((miss.split == 'test').sum()))
assert (miss.split == 'train').sum() == 0, 'some TRAIN audio files were not found'
ALL = ALL[ALL.path.notna()].reset_index(drop=True)      # unreadable/missing test files get a fallback value at submission time
ALL = ALL.set_index('filename', drop=False)
TRAIN_NAMES = ALL.index[ALL.split == 'train']; TEST_NAMES = ALL.index[ALL.split == 'test']
Y = ALL.loc[TRAIN_NAMES, 'y'].values.astype(float)
display(tr.head()) if 'display' in globals() else print(tr.head())


## 5. Exploratory data analysis
Label type, balance and spread are measured, not assumed.

In [ ]:
# ---- 5. Exploratory data analysis: the label ----------------------------------------------------------
# WHY: never assume the label distribution. Type (integer/half-step/continuous), balance and spread decide
# the loss, the stratification scheme, the clipping range and what RMSE a "good" model must beat.
uniq = np.sort(np.unique(Y)); steps = np.unique(np.round(np.diff(uniq), 4))
kind = 'integers' if np.allclose(Y, np.round(Y)) else ('half-steps' if np.allclose(Y * 2, np.round(Y * 2)) else 'continuous')
print(pd.Series(Y).describe().round(3).to_string())
print(f'\nunique values: {len(uniq)} | min step(s): {steps[:5]} | label type: {kind}')
vc = pd.Series(np.round(Y, 2)).value_counts().sort_index()
share = (vc / vc.sum()).round(3)
print('\nscore counts:\n', pd.DataFrame({'n': vc, 'share': share}).T.to_string())
p = vc.values / vc.sum()
print(f'normalized entropy={-(p*np.log(p)).sum()/np.log(len(p)):.3f} (1 = perfectly balanced) | '
      f'majority share={p.max():.2%} | imbalance max/min={vc.max()/vc.min():.1f}x | baseline RMSE (predict mean)={Y.std():.3f}')

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(Y, bins=min(30, max(8, len(uniq))), kde=True, ax=ax[0]); ax[0].set_title('Score histogram + KDE')
ax[1].bar(vc.index.astype(str), vc.values, color='steelblue'); ax[1].set_title('Score value counts (balance check)')
sns.ecdfplot(Y, ax=ax[2]); ax[2].set_title('Empirical CDF of score')
plt.tight_layout(); plt.show()


## 6. Audio validation
Readability, sample rate, channels, duration, md5 (for the leakage audit), train-vs-test duration shift.

In [ ]:
# ---- 6. Audio validation ------------------------------------------------------------------------------
# WHY: catch corrupt/short/odd-format files before spending GPU hours, and get duration/size/md5 for the
# leakage audit (duplicates, duration/metadata leakage, train-vs-test shift).
def audio_meta(path):
    rec = dict(ok=True, err='')
    try:
        info = sf.info(str(path))
        rec.update(sr=info.samplerate, channels=info.channels, frames=info.frames, duration=info.duration, subtype=info.subtype)
        with sf.SoundFile(str(path)) as f:       # decode a chunk from the middle -> catches truncated files
            f.seek(f.frames // 2); chunk = f.read(1600)
            rec['peak_mid'] = float(np.abs(chunk).max()) if len(chunk) else 0.0
    except Exception as e:
        rec.update(ok=False, err=str(e)[:80])
    rec['size'] = os.path.getsize(path)
    with open(path, 'rb') as fh: rec['md5'] = hashlib.md5(fh.read()).hexdigest()
    return rec

META_CACHE = CFG.CACHE / 'audio_meta.pkl'
if META_CACHE.exists():
    meta = pd.read_pickle(META_CACHE)
else:
    meta = pd.DataFrame(Parallel(n_jobs=CFG.N_JOBS, prefer='threads')(delayed(audio_meta)(p) for p in tqdm(ALL.path)), index=ALL.index)
    meta.to_pickle(META_CACHE)
ALL = ALL.join(meta)
print('unreadable files:', int((~ALL.ok).sum()), '| silent-ish mid-chunks:', int((ALL.peak_mid < 1e-4).sum()))
print('sample-rate counts:', ALL.groupby('split').sr.value_counts().to_dict())
print('channels:', ALL.channels.value_counts().to_dict(), '| subtype:', ALL.subtype.value_counts().to_dict())
print(ALL.groupby('split').duration.describe().round(2).to_string())

trn = ALL[ALL.split == 'train']
rho, pv = stats.spearmanr(trn.duration, trn.y)
print(f'Spearman(duration, score) = {rho:.3f} (p={pv:.3g})')
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
sns.histplot(data=ALL, x='duration', hue='split', bins=30, element='step', stat='density', common_norm=False, ax=ax[0]); ax[0].set_title('Duration: train vs test')
ALL.sr.value_counts().plot.bar(ax=ax[1], title='Sample rates')
ax[2].scatter(trn.duration, trn.y, s=8, alpha=.5); ax[2].set_title(f'Score vs duration (rho={rho:.2f})'); ax[2].set_xlabel('s')
sns.boxplot(data=trn.assign(score=trn.y.round(1)), x='score', y='duration', ax=ax[3]); ax[3].set_title('Duration by score')
plt.tight_layout(); plt.show()


## 7. Audio feature extraction
Pause/rate/F0/spectral statistics (CPU, parallel, cached).

In [ ]:
# ---- 7. Audio feature extraction ------------------------------------------------------------------------
# WHY: pauses, rate, F0 dynamics and spectral statistics capture fluency/proficiency, a secondary correlate
# of grammar. CPU-only, label-free (so no leakage), parallel and cached.
def _mask_to_segments(mask, hop_s, min_gap=0.15, min_seg=0.08):
    idx = np.flatnonzero(np.diff(np.concatenate([[0], mask.astype(int), [0]])))
    merged = []
    for s, e in zip(idx[::2], idx[1::2]):
        s, e = s * hop_s, e * hop_s
        if merged and s - merged[-1][1] < min_gap: merged[-1] = (merged[-1][0], e)   # bridge stop closures
        else: merged.append((s, e))
    return [(s, e) for s, e in merged if e - s >= min_seg]

def _agg(out, name, x, fns=('mean', 'std')):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    for fn in fns:
        out[f'{name}_{fn}'] = (getattr(np, fn)(x) if x.size else np.nan)

def extract_audio_features(path, sr=CFG.SR, max_dur=None):
    out = {}
    try:
        y, _ = librosa.load(str(path), sr=sr, mono=True, duration=max_dur)      # max_dur=None -> whole file
    except Exception:
        return {'aud_load_failed': 1.0}
    out['aud_load_failed'] = 0.0
    if len(y) < sr: return out
    hop, n_fft = 160, 512; hop_s = hop / sr; dur = len(y) / sr
    out['aud_dur'] = dur; out['aud_clip_frac'] = float(np.mean(np.abs(y) > 0.99))
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop))
    rms = librosa.feature.rms(S=S, frame_length=n_fft)[0]
    db = librosa.amplitude_to_db(rms + 1e-10)
    p95, p10 = np.percentile(db, 95), np.percentile(db, 10)
    thr = min(max(p95 - 35.0, p10 + 8.0), p95 - 15.0)           # adaptive energy VAD threshold
    speech = db > thr
    out['aud_snr_proxy'] = p95 - p10
    segs = _mask_to_segments(speech, hop_s)
    seg_d = np.array([e - s for s, e in segs]) if segs else np.array([0.0])
    gaps = np.array([segs[i + 1][0] - segs[i][1] for i in range(len(segs) - 1)]) if len(segs) > 1 else np.array([])
    pauses = gaps[gaps >= CFG.PAUSE_MIN_S]
    lead = segs[0][0] if segs else dur; trail = dur - segs[-1][1] if segs else 0.0
    active = max(dur - lead - trail, 1.0); sp_dur = float(seg_d.sum())
    out.update(aud_speech_dur=sp_dur, aud_sil_dur=dur - sp_dur, aud_speech_ratio=sp_dur / dur, aud_lead_sil=lead, aud_trail_sil=trail,
               aud_n_segments=len(segs), aud_seg_dur_mean=seg_d.mean(), aud_seg_dur_max=seg_d.max(),
               aud_n_pauses=len(pauses), aud_pause_rate_pm=len(pauses) / (active / 60),
               aud_n_pauses_500=int((gaps >= .5).sum()), aud_n_pauses_1s=int((gaps >= 1.0).sum()),
               aud_pause_mean=pauses.mean() if len(pauses) else 0.0, aud_pause_max=pauses.max() if len(pauses) else 0.0,
               aud_pause_total=float(pauses.sum()), aud_pause_std=pauses.std() if len(pauses) > 1 else 0.0)
    # syllable-nuclei proxy: prominent smoothed-energy peaks inside speech
    sm = np.convolve(db, np.ones(5) / 5, mode='same')
    pk, _ = find_peaks(sm, prominence=3.0, distance=max(1, int(0.12 / hop_s)))
    n_syl = int(speech[pk].sum())
    out['aud_syl_rate_artic'] = n_syl / max(sp_dur, 1e-3); out['aud_syl_rate_overall'] = n_syl / active

    def sel(a):                                                  # keep speech frames only (silence would dominate stats)
        n = min(a.shape[-1], len(speech)); m = speech[:n]
        b = a[..., :n][..., m]
        return b if b.shape[-1] > 20 else a
    mel = librosa.feature.melspectrogram(S=S ** 2, sr=sr, n_mels=40)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=20)
    dmf = librosa.feature.delta(mfcc, width=9)[:13]
    for i, (a, b) in enumerate(zip(sel(mfcc).mean(1), sel(mfcc).std(1))): out[f'aud_mfcc{i}_mean'] = a; out[f'aud_mfcc{i}_std'] = b
    for i, (a, b) in enumerate(zip(sel(dmf).mean(1), sel(dmf).std(1))): out[f'aud_dmfcc{i}_mean'] = a; out[f'aud_dmfcc{i}_std'] = b
    for nm, arr in (('centroid', librosa.feature.spectral_centroid(S=S, sr=sr)[0]), ('bandwidth', librosa.feature.spectral_bandwidth(S=S, sr=sr)[0]),
                    ('rolloff', librosa.feature.spectral_rolloff(S=S, sr=sr)[0]),
                    ('zcr', librosa.feature.zero_crossing_rate(y, frame_length=n_fft, hop_length=hop)[0]), ('rms', rms)):
        _agg(out, f'aud_{nm}', sel(arr))
    con = sel(librosa.feature.spectral_contrast(S=S, sr=sr, fmin=100.0, n_bands=5))
    for i, v in enumerate(con.mean(1)): out[f'aud_contrast{i}_mean'] = v
    # pitch (F0) - pyin at a coarse hop for speed
    try:
        f0, vflag, _ = librosa.pyin(y, fmin=65, fmax=400, sr=sr, frame_length=1024, hop_length=320)
        v = f0[np.isfinite(f0)]
        out['aud_voiced_ratio'] = float(np.mean(vflag))
        if v.size > 20:
            med = np.median(v); stf = 12 * np.log2(f0 / med)
            out.update(aud_f0_mean=v.mean(), aud_f0_median=med, aud_f0_std=v.std(), aud_f0_min=v.min(), aud_f0_max=v.max(),
                       aud_f0_st_std=np.nanstd(stf), aud_f0_st_range=np.nanpercentile(stf, 95) - np.nanpercentile(stf, 5),
                       aud_f0_dyn=np.nanmean(np.abs(np.diff(stf))))
            idx = np.flatnonzero(np.diff(np.concatenate([[0], vflag.astype(int), [0]])))
            runs = (idx[1::2] - idx[::2]) * 320 / sr
            out['aud_n_voiced_segs'] = len(runs); out['aud_voiced_seg_mean'] = runs.mean() if len(runs) else np.nan
    except Exception:
        pass
    return out

def run_audio_features(max_dur, cache_path):
    """Cached extraction. SAFE by default: sequential execution (librosa.pyin + joblib/loky worker processes has produced a segmentation fault)."""
    if cache_path.exists(): return pd.read_pickle(cache_path)
    paths = list(ALL.path)
    if CFG.AUDIO_PARALLEL == 'sequential':
        res = [extract_audio_features(p, max_dur=max_dur) for p in tqdm(paths, desc=f'audio feats (max_dur={max_dur})')]
    else:
        backend = 'threading' if CFG.AUDIO_PARALLEL == 'threads' else 'loky'
        extract_audio_features(paths[0], max_dur=max_dur)                                  # warm up numba JIT once in the main process
        res = Parallel(n_jobs=CFG.N_JOBS, backend=backend)(delayed(extract_audio_features)(p, max_dur=max_dur) for p in tqdm(paths))
    df = pd.DataFrame(res, index=ALL.index); df.to_pickle(cache_path); return df

aud_df = run_audio_features(None, CFG.CACHE / 'audio_feats_v1.pkl')                       # full-length view (cached from v1)
FEATURE_BLOCKS['audio'] = aud_df
print('audio features (full length):', aud_df.shape, '| failed loads:', int(aud_df.get('aud_load_failed', pd.Series([0])).sum()))


## 8. Speech-to-text
Faster-Whisper with a verbatim-leaning prompt; resumable cache; ASR confidence + word-timing features.

In [ ]:
# ---- 8. Speech-to-text -----------------------------------------------------------------------------------
# WHY: grammar lives in the words. Transcripts are produced ONCE, cached (resumable jsonl) and never recomputed in CV.
def read_jsonl(p):
    if not Path(p).exists(): return []
    out = []
    with open(p) as fh:
        for line in fh:
            try: out.append(json.loads(line))
            except Exception: pass
    return out

def norm_words(text):
    t = str(text).lower().replace('’', "'")
    return re.sub(r"[^a-z0-9' ]+", ' ', t).split()

def word_edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

def run_whisper(items, cache_path):
    from faster_whisper import WhisperModel
    model = WhisperModel(mpath(CFG.WHISPER_MODEL), device=DEVICE, compute_type=CFG.WHISPER_COMPUTE)
    with open(cache_path, 'a') as fh:
        for fname, path in tqdm(items, desc=f'Whisper {CFG.WHISPER_MODEL}'):
            try:
                # decode the audio ourselves (avoids the faster-whisper / PyAV version clash)
                audio, _ = librosa.load(str(path), sr=16000, mono=True)
                segments, info = model.transcribe(audio.astype(np.float32), language='en', task='transcribe', beam_size=5,
                                                  temperature=(0.0, 0.2, 0.4), condition_on_previous_text=False,
                                                  initial_prompt=CFG.WHISPER_PROMPT, word_timestamps=True, vad_filter=False)
                segs = [dict(start=s.start, end=s.end, text=s.text, avg_logprob=s.avg_logprob, no_speech_prob=s.no_speech_prob,
                             compression_ratio=s.compression_ratio,
                             words=[dict(w=w.word, s=w.start, e=w.end, p=w.probability) for w in (s.words or [])]) for s in segments]
                rec = dict(filename=fname, text=' '.join(x['text'].strip() for x in segs), language=info.language,
                           lang_prob=info.language_probability, duration=info.duration, segments=segs)
            except Exception as e:                      # one bad file must not kill a 1-hour run
                print(f'[warn] {fname}: {repr(e)[:150]} -> empty transcript')
                rec = dict(filename=fname, text='', language='en', lang_prob=0.0, duration=0.0, segments=[])
                try:   # PATCH: retry once WITHOUT word timestamps (the IndexError comes from the word-alignment step)
                    segments, info = model.transcribe(audio.astype(np.float32), language='en', task='transcribe', beam_size=5,
                                                      condition_on_previous_text=False, initial_prompt=CFG.WHISPER_PROMPT, word_timestamps=False)
                    segs = [dict(start=s.start, end=s.end, text=s.text, avg_logprob=s.avg_logprob, no_speech_prob=s.no_speech_prob,
                                 compression_ratio=s.compression_ratio, words=[]) for s in segments]
                    rec = dict(filename=fname, text=' '.join(x['text'].strip() for x in segs), language=info.language,
                               lang_prob=info.language_probability, duration=info.duration, segments=segs)
                except Exception: pass
            fh.write(json.dumps(rec) + '\n'); fh.flush()      # flush each file -> safe to resume after a Kaggle timeout

WHISPER_CACHE = CFG.CACHE / f'whisper_{tag(CFG.WHISPER_MODEL)}.jsonl'
ASR = {r['filename']: r for r in read_jsonl(WHISPER_CACHE)}
todo = [(f, p) for f, p in zip(ALL.filename, ALL.path) if f not in ASR]
print(f'cached transcripts: {len(ASR)} | to do: {len(todo)}')
if todo and CFG.USE_WHISPER:
    run_whisper(todo, WHISPER_CACHE)
    ASR = {r['filename']: r for r in read_jsonl(WHISPER_CACHE)}
missing = [f for f in ALL.filename if f not in ASR]
if missing:
    raise RuntimeError(f'{len(missing)} transcripts missing. Enable USE_WHISPER (GPU + internet) or attach the cached '
                       f'{WHISPER_CACHE.name} as a dataset into {CFG.CACHE}.')

def clean_asr_text(t):
    t = str(t).replace(CFG.WHISPER_PROMPT, ' ')               # drop prompt echoes (happens on silence)
    return re.sub(r'\s+', ' ', t).strip()
TEXT = pd.Series({f: clean_asr_text(ASR[f]['text']) for f in ALL.filename})

def asr_features(rec, audio_dur):
    out = {}; segs = rec['segments']; words = [w for s in segs for w in s['words']]
    n = len(norm_words(rec['text']))
    out['asr_n_segments'] = len(segs); out['asr_lang_prob'] = rec.get('lang_prob', np.nan)
    if segs:
        d = np.array([max(s['end'] - s['start'], 1e-3) for s in segs])
        out['asr_logprob_mean'] = float(np.average([s['avg_logprob'] for s in segs], weights=d))
        out['asr_logprob_min'] = min(s['avg_logprob'] for s in segs)
        out['asr_nospeech_max'] = max(s['no_speech_prob'] for s in segs)
        out['asr_compress_max'] = max(s['compression_ratio'] for s in segs)
    if words:
        p = np.array([w['p'] for w in words]); wd = np.array([max(w['e'] - w['s'], 0) for w in words])
        gaps = np.array([max(words[i + 1]['s'] - words[i]['e'], 0) for i in range(len(words) - 1)]) if len(words) > 1 else np.array([0.0])
        span = max(words[-1]['e'] - words[0]['s'], 1e-3)
        out.update(asr_word_p_mean=p.mean(), asr_word_p_p10=np.percentile(p, 10), asr_word_p_lt50=float(np.mean(p < .5)),
                   asr_word_dur_mean=wd.mean(), asr_word_dur_std=wd.std(), asr_gap_mean=gaps.mean(), asr_gap_std=gaps.std(),
                   asr_gap_max=gaps.max(), asr_gaps_300=int((gaps >= .3).sum()), asr_gaps_500=int((gaps >= .5).sum()),
                   asr_gaps_1s=int((gaps >= 1.0).sum()), asr_speech_start=words[0]['s'], asr_speech_end=words[-1]['e'],
                   asr_coverage=span / max(audio_dur, 1e-3), asr_wpm_span=n / (span / 60),
                   asr_artic_wpm=n / ((wd.sum() + gaps[gaps < .3].sum()) / 60 + 1e-6))
    out['asr_wpm_total'] = n / (max(audio_dur, 1e-3) / 60)
    return out

asr_df = pd.DataFrame([asr_features(ASR[f], ALL.loc[f, 'duration']) for f in ALL.filename], index=ALL.index)
FEATURE_BLOCKS['asr'] = asr_df
print('ASR features:', asr_df.shape)

### 9. Literal CTC transcript, CTC confidence, SSL audio embeddings

In [ ]:
# ---- 9. Literal (LM-free) CTC transcript + SSL audio embeddings + CTC confidence -------------------------
# WHY: wav2vec2-CTC has no strong language model, so it does NOT repair grammar the way Whisper does.
# (1) Whisper-vs-literal disagreement is a grammar-repair signal; (2) CTC confidence ~ pronunciation quality;
# (3) mid-layer hidden states are strong frozen audio representations (and early layers encode speaker/channel,
# used for the speaker-overlap audit).
def run_w2v(paths, max_dur=None):
    from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
    name = mpath(CFG.W2V_MODEL)
    proc = Wav2Vec2Processor.from_pretrained(name)
    model = Wav2Vec2ForCTC.from_pretrained(name).to(DEVICE).eval()
    if HAS_GPU: model = model.half()
    blank = proc.tokenizer.pad_token_id; texts, scal, embs = [], [], defaultdict(list)
    for p in tqdm(paths, desc='wav2vec2'):
        y, _ = librosa.load(str(p), sr=16000, mono=True, duration=max_dur)
        iv = proc(y, sampling_rate=16000, return_tensors='pt').input_values.to(DEVICE)
        if HAS_GPU: iv = iv.half()
        with torch.no_grad(): o = model(iv, output_hidden_states=True)
        lg = o.logits[0].float(); pr = torch.softmax(lg, -1); mx, ids = pr.max(-1)
        texts.append(proc.batch_decode(ids[None])[0].lower())
        nb = ids != blank; ent = -(pr * torch.log(pr + 1e-9)).sum(-1)
        scal.append(dict(ctc_conf_mean=float(mx[nb].mean()) if nb.any() else np.nan, ctc_conf_p10=float(torch.quantile(mx[nb], .1)) if nb.sum() > 5 else np.nan,
                         ctc_blank_ratio=float((~nb).float().mean()), ctc_entropy_mean=float(ent.mean()), ctc_nonblank_entropy=float(ent[nb].mean()) if nb.any() else np.nan))
        hs = o.hidden_states; L = len(hs) - 1
        for tagname, l in (('early', L // 4), ('mid', L // 2), ('late', (3 * L) // 4)):
            h = hs[l][0].float()
            embs[f'{tagname}_mean'].append(h.mean(0).cpu().numpy())
            if tagname != 'early': embs[f'{tagname}_std'].append(h.std(0).cpu().numpy())
    return texts, pd.DataFrame(scal), {k: np.stack(v) for k, v in embs.items()}

W2V_TXT = CFG.CACHE / f'w2v_{tag(CFG.W2V_MODEL)}.pkl'
W2V = None
if W2V_TXT.exists():
    W2V = pd.read_pickle(W2V_TXT)
elif CFG.USE_W2V and torch is not None:
    try:
        texts, sc, embs = run_w2v(list(ALL.path))
        W2V = dict(texts=texts, scalars=sc, embs=embs); pd.to_pickle(W2V, W2V_TXT)
    except Exception as e:
        print('wav2vec2 pass skipped:', repr(e)[:200])
if W2V is not None:
    lit = pd.Series(W2V['texts'], index=ALL.index)
    ctc = W2V['scalars'].set_axis(ALL.index)
    dis = []
    for f in ALL.index:
        a, b = norm_words(TEXT[f]), norm_words(lit[f])
        d = word_edit_distance(a, b) if a and b else np.nan
        dis.append(dict(asr_wer_whisper_vs_literal=d / max(len(a), 1), asr_len_ratio_literal=len(b) / max(len(a), 1)))
    FEATURE_BLOCKS['ctc'] = ctc
    FEATURE_BLOCKS['asr_disagree'] = pd.DataFrame(dis, index=ALL.index)
    EMB_BLOCKS['embw2v'] = np.hstack([W2V['embs']['mid_mean'], W2V['embs']['mid_std'], W2V['embs']['late_mean']])
    SPK_EMB = W2V['embs']['early_mean']
    print('wav2vec2 features:', ctc.shape, '| audio embedding dim:', EMB_BLOCKS['embw2v'].shape[1])
else:
    SPK_EMB = None
    print('No wav2vec2 outputs (USE_W2V=False or torch missing): continuing without CTC/SSL features.')


## 10. Transcript analysis

In [ ]:
# ---- 10. Transcript analysis ---------------------------------------------------------------------------------
# WHY: audit ASR quality BEFORE modelling: empty/hallucinated transcripts, language mismatches, and whether
# transcript length behaves sensibly vs duration and score.
tw = TEXT.map(lambda t: len(norm_words(t)))
chk = pd.DataFrame({'n_words': tw, 'dur': ALL.duration, 'split': ALL.split, 'y': ALL.y,
                    'compress': asr_df.get('asr_compress_max'), 'lang_p': asr_df.get('asr_lang_prob')})
print('empty/very short (<20 words):', int((tw < 20).sum()), '| hallucination-suspect (compression>2.4):', int((chk.compress > 2.4).sum()),
      '| low English prob (<0.9):', int((chk.lang_p < 0.9).sum()))
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(data=chk, x='n_words', hue='split', bins=30, element='step', ax=ax[0]); ax[0].set_title('Words per transcript')
ax[1].scatter(chk.dur, chk.n_words, s=8, alpha=.5); ax[1].set_title('Words vs duration')
t_ = chk[chk.split == 'train']; ax[2].scatter(t_.n_words, t_.y, s=8, alpha=.5); ax[2].set_title(f"Score vs words (rho={stats.spearmanr(t_.n_words, t_.y)[0]:.2f})")
plt.tight_layout(); plt.show()
for lvl in np.unique(np.round(Y)):
    f = ALL[(ALL.split == 'train') & (np.round(ALL.y) == lvl)].filename.iloc[0]
    print(f'\n[score {ALL.loc[f, "y"]}] {TEXT[f][:350]}')

# Score-0 recordings (about 5% of train): are they silent / non-speech? If so, speech-rate + no-speech features gate them.
zt = chk[chk.split == 'train'].assign(zero=lambda d: d.y == 0)
print('\nMedian words / hallucination-compression by (score == 0):\n', zt.groupby('zero')[['n_words', 'dur', 'compress']].median().round(2).to_string())
print('score-0 files with < 20 words:', int(((zt.zero) & (zt.n_words < 20)).sum()), 'of', int(zt.zero.sum()))


## 11. Linguistic feature engineering
Disfluency, lexical, syntax (spaCy), heuristic error counters, CoLA acceptability, LM surprisal, GEC edit features, LanguageTool.

In [ ]:
# ---- 11. Linguistic feature engineering -------------------------------------------------------------------
# WHY: the rubric is about morphosyntax. We build many *noisy but diverse* detectors; the supervised model learns
# how much to trust each (no grammar checker is trusted blindly - outputs are just numeric features).

FILLERS = {
    'um', 'umm', 'uh', 'uhh', 'uhm', 'er', 'erm',
    'ah', 'hmm', 'hm', 'mm', 'mhm', 'eh'
}

DISCOURSE = [
    'you know', 'i mean', 'kind of', 'sort of', 'like',
    'basically', 'actually', 'literally', 'well', 'so',
    'right', 'okay'
]

SELF_CORR = [
    r'\bi mean\b',
    r'\bsorry\b',
    r'\bor rather\b',
    r'\bno wait\b',
    r'\bwait\b',
    r'\blet me (?:rephrase|start)\b'
]

ARTICLES = {'a', 'an', 'the'}

PREPS = set(
    'in on at by for with about against between into through during '
    'before after above below to from up down of off over under near '
    'across along among around beyond despite except inside outside '
    'since than toward towards until upon within without'.split()
)

AUXS = set(
    'is are was were am be been being do does did have has had will '
    'would can could should may might must shall'.split()
)

MASS = set(
    'water time money music work information food life love english '
    'school home college family health weather advice news homework '
    'knowledge traffic education research experience happiness society '
    'nature technology'.split()
)


def split_sentences(t):
    """
    Split transcript into non-empty sentences.
    """
    if not isinstance(t, str):
        t = '' if t is None else str(t)

    t = t.strip()

    if not t:
        return []

    return [
        s.strip()
        for s in re.split(r'(?<=[.!?])\s+|\n+', t)
        if s.strip()
    ]


def mattr(w, win=50):
    """
    Moving average type-token ratio.
    """
    if len(w) < win:
        return len(set(w)) / max(len(w), 1)

    return float(
        np.mean([
            len(set(w[i:i + win])) / win
            for i in range(0, len(w) - win + 1, 5)
        ])
    )


try:
    from wordfreq import zipf_frequency
except Exception:
    zipf_frequency = None


def text_features(t, dur):
    """
    Basic lexical, fluency, repetition and sentence-level linguistic features.

    Important:
    Empty transcripts are handled explicitly so indexing s[0] / s[-1]
    never raises IndexError.
    """

    # ------------------------------------------------------------------
    # Safe input handling
    # ------------------------------------------------------------------
    if not isinstance(t, str):
        t = '' if t is None else str(t)

    try:
        dur = float(dur)
    except Exception:
        dur = 0.0

    # ------------------------------------------------------------------
    # Words
    # ------------------------------------------------------------------
    w = norm_words(t)
    n = max(len(w), 1)

    low = ' ' + ' '.join(w) + ' '

    # ------------------------------------------------------------------
    # Sentences
    # ------------------------------------------------------------------
    sents = split_sentences(t)

    # IMPORTANT:
    # Do NOT use [''] because later s[0] and s[-1] would fail.
    # Instead keep a separate empty-transcript case.
    has_sentences = len(sents) > 0

    if has_sentences:
        sl = np.array(
            [len(norm_words(s)) for s in sents],
            dtype=float
        )
    else:
        # Empty array is safe because we explicitly handle all reductions
        # below.
        sl = np.array([], dtype=float)

    # ------------------------------------------------------------------
    # Output dictionary
    # ------------------------------------------------------------------
    o = {}

    # ------------------------------------------------------------------
    # Sentence statistics
    # ------------------------------------------------------------------
    if has_sentences:
        sent_len_mean = float(sl.mean())
        sent_len_med = float(np.median(sl))
        sent_len_max = float(sl.max())
        sent_len_std = float(sl.std())

        short_sent_rate = float(np.mean(sl <= 3))
        long_sent_rate = float(np.mean(sl > 35))

        # Safe because sents is non-empty
        and_chain = max(
            f' {s.lower()} '.count(' and ')
            for s in sents
        )

        comma_per_sent = t.count(',') / len(sents)
        sent_per_100w = 100 * len(sents) / n

        # These were causing the IndexError on empty strings
        term_punct_ok = float(
            np.mean([s[-1] in '.!?' for s in sents])
        )

        cap_ok = float(
            np.mean([s[0].isupper() for s in sents])
        )

    else:
        # Empty transcript defaults
        sent_len_mean = 0.0
        sent_len_med = 0.0
        sent_len_max = 0.0
        sent_len_std = 0.0

        short_sent_rate = 0.0
        long_sent_rate = 0.0
        and_chain = 0.0

        comma_per_sent = 0.0
        sent_per_100w = 0.0

        term_punct_ok = 0.0
        cap_ok = 0.0

    # ------------------------------------------------------------------
    # Basic lexical features
    # ------------------------------------------------------------------
    o.update(
        txt_n_words=len(w),
        txt_n_sent=len(sents),

        txt_sent_len_mean=sent_len_mean,
        txt_sent_len_med=sent_len_med,
        txt_sent_len_max=sent_len_max,
        txt_sent_len_std=sent_len_std,

        txt_n_unique=len(set(w)),
        txt_unique_ratio=len(set(w)) / n,
        txt_root_ttr=len(set(w)) / math.sqrt(n),

        txt_mattr=mattr(w),

        txt_wlen_mean=(
            float(np.mean([len(x) for x in w]))
            if w else 0.0
        ),

        txt_n_long=sum(len(x) >= 7 for x in w),

        txt_long_ratio=(
            sum(len(x) >= 7 for x in w) / n
        ),

        txt_n_repeated=len(w) - len(set(w)),

        txt_wpm=(
            len(w) / (dur / 60 + 1e-6)
            if dur > 0
            else 0.0
        )
    )

    # ------------------------------------------------------------------
    # Disfluency / fillers
    # ------------------------------------------------------------------
    fill = sum(x in FILLERS for x in w)

    o.update(
        txt_fillers=fill,

        txt_filler_rate=100 * fill / n,

        txt_um=sum(
            x in {'um', 'umm', 'uhm'}
            for x in w
        ),

        txt_uh=sum(
            x in {'uh', 'uhh', 'er', 'erm'}
            for x in w
        ),

        txt_like=w.count('like'),

        txt_discourse_rate=(
            100 * sum(
                low.count(f' {m} ')
                for m in DISCOURSE
            ) / n
        ),

        txt_rep_word=len(
            re.findall(
                r'\b(\w+)(?:\s+\1\b)+',
                low
            )
        ),

        txt_rep_phrase=len(
            re.findall(
                r'\b(\w+\s+\w+)\s+\1\b',
                low
            )
        ),

        txt_rep_punct=len(
            re.findall(
                r'\b(\w+)[,\s\.\-—…]+\1\b',
                t.lower()
            )
        ),

        txt_false_start=(
            len(re.findall(r'(—|--|\.\.\.|…)', t))
            +
            len(re.findall(r'\b\w+-\s', t))
        ),

        txt_selfcorr=sum(
            len(re.findall(p, low))
            for p in SELF_CORR
        ),

        txt_hesitation=fill
    )

    # ------------------------------------------------------------------
    # Repetition rate
    # ------------------------------------------------------------------
    o['txt_rep_rate'] = (
        100
        * (
            o['txt_rep_word']
            + o['txt_rep_phrase']
        )
        / n
    )

    # ------------------------------------------------------------------
    # Sentence quality / structure features
    # ------------------------------------------------------------------
    o.update(
        txt_term_punct_ok=term_punct_ok,
        txt_cap_ok=cap_ok,

        txt_short_sent_rate=short_sent_rate,
        txt_long_sent_rate=long_sent_rate,

        txt_and_chain=and_chain,

        txt_comma_per_sent=comma_per_sent,

        txt_sent_per_100w=sent_per_100w
    )

    # ------------------------------------------------------------------
    # Word frequency features
    # ------------------------------------------------------------------
    if zipf_frequency is not None and w:

        z = np.array([
            zipf_frequency(x, 'en')
            for x in w
            if len(x) > 3
        ] or [np.nan])

        o.update(
            txt_zipf_mean=float(np.nanmean(z)),
            txt_zipf_p10=float(np.nanpercentile(z, 10)),
            txt_rare_rate=float(np.nanmean(z < 3.0))
        )

    else:
        # Keep columns present even when wordfreq is unavailable
        o.update(
            txt_zipf_mean=np.nan,
            txt_zipf_p10=np.nan,
            txt_rare_rate=np.nan
        )

    return o


# ----------------------------------------------------------------------
# Build text feature dataframe
# ----------------------------------------------------------------------
txt_df = pd.DataFrame(
    [
        text_features(
            TEXT[f],
            ALL.loc[f, 'duration']
        )
        for f in ALL.index
    ],
    index=ALL.index
)

FEATURE_BLOCKS['txt'] = txt_df

SENTS = {
    f: split_sentences(TEXT[f])
    for f in ALL.index
}

print('basic/disfluency text features:', txt_df.shape)


# ---- spaCy: syntax complexity + heuristic error counters ------------------------------

A_EXC = {
    'one', 'once', 'university', 'union', 'unique', 'useful',
    'user', 'usual', 'european', 'uniform', 'united', 'unit',
    'use', 'used', 'usually'
}

AN_EXC = {
    'hour', 'hours', 'honest', 'honor', 'honour',
    'heir', 'mba', 'fbi', 'hourly'
}

COLLECTIVE = {
    'people', 'police', 'staff', 'team',
    'family', 'government', 'group', 'class'
}

SING_PLURAL_OK = {
    'news', 'physics', 'series', 'means',
    'mathematics', 'economics', 'politics'
}


def spacy_features(doc):

    o = {}

    toks = [
        t for t in doc
        if not t.is_space
    ]

    nt = max(len(toks), 1)

    sents = list(doc.sents)
    ns = max(len(sents), 1)

    # ------------------------------------------------------------------
    # Syntactic tree depth
    # ------------------------------------------------------------------
    depths = []

    for s in sents:

        d = 0

        for t in s:

            k = 0
            x = t

            while x.head is not x and k < 60:
                x = x.head
                k += 1

            d = max(d, k)

        depths.append(d)

    o['syn_depth_mean'] = (
        float(np.mean(depths))
        if depths
        else 0
    )

    o['syn_depth_max'] = (
        max(depths)
        if depths
        else 0
    )

    # ------------------------------------------------------------------
    # Dependency distance
    # ------------------------------------------------------------------
    o['syn_depdist_mean'] = float(
        np.mean([
            abs(t.i - t.head.i)
            for t in toks
            if t.head is not t
        ] or [0])
    )

    # ------------------------------------------------------------------
    # Subordination / coordination
    # ------------------------------------------------------------------
    sub = {
        'advcl',
        'ccomp',
        'xcomp',
        'acl',
        'relcl',
        'csubj',
        'csubjpass'
    }

    o['syn_sub_per_sent'] = (
        sum(t.dep_ in sub for t in toks)
        / ns
    )

    o['syn_coord_per_sent'] = (
        sum(t.dep_ == 'conj' for t in toks)
        / ns
    )

    o['syn_passive_rate'] = (
        sum(
            t.dep_ in ('nsubjpass', 'auxpass')
            for t in toks
        )
        / ns
    )

    # ------------------------------------------------------------------
    # POS ratios
    # ------------------------------------------------------------------
    for p in (
        'NOUN',
        'VERB',
        'ADJ',
        'ADV',
        'ADP',
        'DET',
        'PRON',
        'AUX',
        'CCONJ',
        'SCONJ',
        'PROPN',
        'INTJ'
    ):

        o[f'syn_pos_{p.lower()}'] = (
            sum(t.pos_ == p for t in toks)
            / nt
        )

    # ------------------------------------------------------------------
    # Tense
    # ------------------------------------------------------------------
    past = sum(
        t.tag_ == 'VBD'
        for t in toks
    )

    pres = sum(
        t.tag_ in ('VBP', 'VBZ')
        for t in toks
    )

    o['syn_past_ratio'] = (
        past / max(past + pres, 1)
    )

    o['syn_tense_mix'] = (
        min(past, pres)
        / max(past + pres, 1)
    )

    o['syn_modal_rate'] = (
        sum(t.tag_ == 'MD' for t in toks)
        / nt
    )

    # ------------------------------------------------------------------
    # Perfect / progressive
    # ------------------------------------------------------------------
    o['syn_perfect'] = (
        sum(
            t.lemma_ == 'have'
            and t.dep_ == 'aux'
            and t.head.tag_ == 'VBN'
            for t in toks
        )
        / ns
    )

    o['syn_progressive'] = (
        sum(
            t.lemma_ == 'be'
            and t.dep_ == 'aux'
            and t.head.tag_ == 'VBG'
            for t in toks
        )
        / ns
    )

    # ------------------------------------------------------------------
    # Fragment rate
    # ------------------------------------------------------------------
    fin = {
        'VBD',
        'VBP',
        'VBZ',
        'MD'
    }

    long_s = [
        s
        for s in sents
        if len(s) >= 3
    ]

    o['syn_fragment_rate'] = (
        np.mean([
            not any(
                t.tag_ in fin
                for t in s
            )
            for s in long_s
        ])
        if long_s
        else 0
    )

    # ------------------------------------------------------------------
    # Heuristic grammar error counters
    # ------------------------------------------------------------------
    sv = 0
    art = 0
    num = 0
    vform = 0

    for t in toks:

        tl = t.lower_
        h = t.head

        # --------------------------------------------------------------
        # Subject-verb agreement
        # --------------------------------------------------------------
        if (
            t.dep_ == 'nsubj'
            and h.pos_ in ('VERB', 'AUX')
        ):

            if (
                tl in ('he', 'she', 'it')
                and (
                    h.tag_ == 'VBP'
                    or h.lower_ in ('are', 'am')
                )
            ):
                sv += 1

            elif (
                tl == 'i'
                and (
                    h.tag_ == 'VBZ'
                    or h.lower_ == 'are'
                )
            ):
                sv += 1

            elif (
                tl in ('we', 'they', 'you')
                and (
                    h.tag_ == 'VBZ'
                    or (
                        h.lower_ == 'was'
                        and tl != 'you'
                    )
                    or h.lower_ == 'am'
                )
            ):
                sv += 1

            elif (
                t.tag_ == 'NNS'
                and tl not in SING_PLURAL_OK
                and (
                    h.tag_ == 'VBZ'
                    or h.lower_ == 'was'
                )
            ):
                sv += 1

            elif (
                t.tag_ == 'NN'
                and tl not in COLLECTIVE
                and (
                    h.tag_ == 'VBP'
                    or h.lower_ == 'were'
                )
            ):
                sv += 1

        # --------------------------------------------------------------
        # Article errors
        # --------------------------------------------------------------
        if (
            tl == 'a'
            and t.i + 1 < len(doc)
        ):

            nx = doc[t.i + 1].lower_

            if (
                nx
                and nx[0] in 'aeiou'
                and nx not in A_EXC
            ):
                art += 1

        if (
            tl == 'an'
            and t.i + 1 < len(doc)
        ):

            nx = doc[t.i + 1].lower_

            if (
                nx
                and nx[0].isalpha()
                and nx[0] not in 'aeiou'
                and nx not in AN_EXC
            ):
                art += 1

        # --------------------------------------------------------------
        # Number agreement
        # --------------------------------------------------------------
        if (
            tl in ('a', 'an')
            and h.tag_ == 'NNS'
        ):
            num += 1

        if (
            tl in (
                'many',
                'few',
                'several',
                'these',
                'those',
                'multiple',
                'numerous',
                'both'
            )
            and h.tag_ == 'NN'
            and t.dep_ in (
                'det',
                'amod',
                'nummod',
                'quantmod'
            )
        ):
            num += 1

        if (
            tl in (
                'this',
                'that',
                'another',
                'each',
                'every'
            )
            and t.dep_ == 'det'
            and h.tag_ == 'NNS'
        ):
            num += 1

        if (
            tl == 'much'
            and h.tag_ == 'NNS'
        ):
            num += 1

        # --------------------------------------------------------------
        # Verb form errors
        # --------------------------------------------------------------
        if (
            t.tag_ == 'MD'
            and h.tag_ in (
                'VBD',
                'VBZ',
                'VBG',
                'VBN',
                'VBP'
            )
        ):
            vform += 1

        if (
            t.tag_ == 'TO'
            and t.dep_ == 'aux'
            and h.tag_ in (
                'VBD',
                'VBZ'
            )
        ):
            vform += 1

        if (
            t.lemma_ == 'do'
            and t.dep_ == 'aux'
            and h.tag_ in (
                'VBD',
                'VBZ',
                'VBG'
            )
        ):
            vform += 1

        if (
            t.lemma_ == 'have'
            and t.dep_ == 'aux'
            and h.tag_ == 'VBD'
        ):
            vform += 1

    # ------------------------------------------------------------------
    # Missing determiners
    # ------------------------------------------------------------------
    nn = [
        t
        for t in toks
        if (
            t.tag_ == 'NN'
            and t.dep_ in (
                'dobj',
                'pobj',
                'nsubj',
                'attr'
            )
            and t.lower_ not in MASS
        )
    ]

    miss = sum(
        not any(
            c.dep_ in (
                'det',
                'poss',
                'nummod',
                'predet'
            )
            for c in t.children
        )
        for t in nn
    )

    # ------------------------------------------------------------------
    # Final syntax feature dictionary
    # ------------------------------------------------------------------
    o.update(
        syn_err_sv=100 * sv / nt,

        syn_err_article=100 * art / nt,

        syn_err_number=100 * num / nt,

        syn_err_verbform=100 * vform / nt,

        syn_missing_det=(
            miss / max(len(nn), 1)
        ),

        syn_err_total=(
            100 * (sv + art + num + vform)
            / nt
        )
    )

    return o


# ----------------------------------------------------------------------
# spaCy block
# ----------------------------------------------------------------------
syn_df = None

if CFG.USE_SPACY:

    try:

        # PATCH: a half-imported cupy (from the pip install in the same session) makes `import spacy` fail. Purge it so thinc falls back to CPU.

        for _k in [k for k in sys.modules if k == 'cupy' or k.startswith('cupy.')]: del sys.modules[_k]

        sys.modules['cupy'] = None

        import spacy

        nlp = spacy.load(
            CFG.SPACY_MODEL,
            disable=['ner']
        )

        docs = nlp.pipe(
            [
                TEXT[f] if TEXT[f] else '.'
                for f in ALL.index
            ],
            batch_size=16
        )

        syn_df = pd.DataFrame(
            [
                spacy_features(d)
                for d in tqdm(
                    docs,
                    total=len(ALL)
                )
            ],
            index=ALL.index
        )

        FEATURE_BLOCKS['syn'] = syn_df

        print(
            'spaCy features:',
            syn_df.shape
        )

    except Exception as e:

        print(
            'spaCy features skipped:',
            repr(e)[:160]
        )


# ---- Neural grammar detectors (cached) ------------------------------------------------


def cached(name, fn):

    p = CFG.CACHE / f'{name}.pkl'

    if p.exists():
        return pd.read_pickle(p)

    try:

        r = fn()

        pd.to_pickle(
            r,
            p
        )

        return r

    except Exception as e:

        print(
            f'[skip {name}]',
            repr(e)[:200]
        )

        return None


# ----------------------------------------------------------------------
# CoLA acceptability
# ----------------------------------------------------------------------
def cola_block():

    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification
    )

    tok = AutoTokenizer.from_pretrained(
        mpath(CFG.COLA_MODEL)
    )

    m = AutoModelForSequenceClassification.from_pretrained(
        mpath(CFG.COLA_MODEL)
    ).to(DEVICE).eval()

    flat = [
        (f, s)
        for f in ALL.index
        for s in SENTS[f]
        if len(norm_words(s)) >= 3
    ]

    pr = np.zeros(
        len(flat),
        dtype=float
    )

    for b in tqdm(
        range(0, len(flat), 64),
        desc='CoLA'
    ):

        batch = flat[b:b + 64]

        enc = tok(
            [s for _, s in batch],
            padding=True,
            truncation=True,
            max_length=96,
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():

            logits = m(**enc).logits.float()

            pr[b:b + 64] = (
                torch.softmax(
                    logits,
                    -1
                )[:, 1]
                .cpu()
                .numpy()
            )

    d = defaultdict(list)

    for (f, s), p in zip(
        flat,
        pr
    ):
        d[f].append(
            (
                p,
                len(norm_words(s))
            )
        )

    rows = []

    for f in ALL.index:

        v = np.array(
            d.get(f)
            or [(np.nan, 1)]
        )

        p = v[:, 0]
        l = v[:, 1]

        rows.append(
            dict(
                cola_p1_mean=np.nanmean(p),
                cola_p1_min=np.nanmin(p),
                cola_p1_q25=np.nanpercentile(
                    p,
                    25
                ),
                cola_p1_std=np.nanstd(p),
                cola_frac_bad=np.nanmean(
                    p < .5
                ),
                cola_len_weighted=(
                    np.nansum(p * l)
                    / max(l.sum(), 1)
                )
            )
        )

    return pd.DataFrame(
        rows,
        index=ALL.index
    )


# ----------------------------------------------------------------------
# LM surprisal
# ----------------------------------------------------------------------
def lm_block():

    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM
    )

    tok = AutoTokenizer.from_pretrained(
        mpath(CFG.LM_MODEL)
    )

    lm = AutoModelForCausalLM.from_pretrained(
        mpath(CFG.LM_MODEL)
    ).to(DEVICE).eval()

    rows = []

    for f in tqdm(
        ALL.index,
        desc='LM surprisal'
    ):

        text = TEXT[f] or '.'

        ids = tok(
            text,
            return_tensors='pt',
            truncation=True,
            max_length=1000
        ).input_ids.to(DEVICE)

        # --------------------------------------------------------------
        # Safely handle models without BOS token
        # --------------------------------------------------------------
        if tok.bos_token_id is not None:

            bos = torch.tensor(
                [[tok.bos_token_id]],
                device=DEVICE
            )

            ids = torch.cat(
                [bos, ids],
                dim=1
            )

        # Need at least 2 tokens to calculate token-level surprisal
        if ids.shape[1] < 2:

            rows.append(
                dict(
                    lm_nll_mean=0.0,
                    lm_nll_std=0.0,
                    lm_nll_p90=0.0,
                    lm_nll_max=0.0,
                    lm_frac_gt8=0.0,
                    lm_ppl=1.0
                )
            )

            continue

        with torch.no_grad():

            lg = lm(
                ids
            ).logits[0, :-1].float()

        nll = (
            torch.nn.functional.cross_entropy(
                lg,
                ids[0, 1:],
                reduction='none'
            )
            .cpu()
            .numpy()
        )

        rows.append(
            dict(
                lm_nll_mean=float(nll.mean()),
                lm_nll_std=float(nll.std()),
                lm_nll_p90=float(
                    np.percentile(nll, 90)
                ),
                lm_nll_max=float(nll.max()),
                lm_frac_gt8=float(
                    np.mean(nll > 8)
                ),
                lm_ppl=float(
                    math.exp(
                        min(
                            nll.mean(),
                            20
                        )
                    )
                )
            )
        )

    return pd.DataFrame(
        rows,
        index=ALL.index
    )


# ----------------------------------------------------------------------
# GEC block
# ----------------------------------------------------------------------
def gec_block():

    from transformers import (
        AutoTokenizer,
        AutoModelForSeq2SeqLM
    )

    tok = AutoTokenizer.from_pretrained(
        mpath(CFG.GEC_MODEL)
    )

    m = AutoModelForSeq2SeqLM.from_pretrained(
        mpath(CFG.GEC_MODEL)
    ).to(DEVICE).eval()

    flat = [
        (f, s)
        for f in ALL.index
        for s in SENTS[f]
    ]

    order = np.argsort(
        [len(s) for _, s in flat]
    )

    fixed = [None] * len(flat)

    for b in tqdm(
        range(0, len(order), 32),
        desc='GEC'
    ):

        ix = order[b:b + 32]

        enc = tok(
            [
                CFG.GEC_PREFIX + flat[i][1]
                for i in ix
            ],
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=128
        ).to(DEVICE)

        with torch.no_grad():

            out = m.generate(
                **enc,
                max_new_tokens=128,
                num_beams=4
            )

        decoded = tok.batch_decode(
            out,
            skip_special_tokens=True
        )

        for i, s in zip(
            ix,
            decoded
        ):
            fixed[i] = s

    per = defaultdict(list)

    for (f, s), c in zip(
        flat,
        fixed
    ):
        per[f].append(
            (
                s,
                c
            )
        )

    rows = []

    for f in ALL.index:

        tot = 0
        ed = 0
        ch = 0
        ins = 0
        dele = 0
        rep = 0
        art = 0
        prp = 0
        aux = 0
        mor = 0
        wch = 0
        oth = 0

        mx = 0.0
        cw = 0

        for s, c in per.get(
            f,
            []
        ):

            a = norm_words(s)
            b = norm_words(c)

            tot += len(a)
            cw += len(b)

            e_s = 0

            for (
                tg,
                i1,
                i2,
                j1,
                j2
            ) in difflib.SequenceMatcher(
                a=a,
                b=b,
                autojunk=False
            ).get_opcodes():

                if tg == 'equal':
                    continue

                x = a[i1:i2]
                y = b[j1:j2]

                k = max(
                    len(x),
                    len(y)
                )

                e_s += k

                allt = set(x) | set(y)

                if tg == 'insert':
                    ins += k

                elif tg == 'delete':
                    dele += k

                else:
                    rep += k

                if (
                    allt
                    and allt <= ARTICLES
                ):
                    art += k

                elif (
                    allt
                    and allt <= PREPS
                ):
                    prp += k

                elif (
                    allt
                    and allt <= AUXS
                ):
                    aux += k

                elif (
                    tg == 'replace'
                    and len(x) == len(y) == 1
                    and (
                        difflib.SequenceMatcher(
                            a=x[0],
                            b=y[0]
                        ).ratio() >= .6
                    )
                ):
                    mor += k

                elif tg == 'replace':
                    wch += k

                else:
                    oth += k

            ed += e_s

            ch += (
                e_s > 0
            )

            mx = max(
                mx,
                e_s / max(
                    len(a),
                    1
                )
            )

        n = max(
            tot,
            1
        )

        rows.append(
            dict(
                gec_edit_rate=100 * ed / n,

                gec_frac_changed=(
                    ch
                    / max(
                        len(per.get(f, [])),
                        1
                    )
                ),

                gec_max_sent_rate=mx,

                gec_ins=100 * ins / n,

                gec_del=100 * dele / n,

                gec_rep=100 * rep / n,

                gec_article=100 * art / n,

                gec_prep=100 * prp / n,

                gec_aux=100 * aux / n,

                gec_morph=100 * mor / n,

                gec_wordchoice=100 * wch / n,

                gec_other=100 * oth / n,

                gec_len_ratio=cw / n
            )
        )

    return pd.DataFrame(
        rows,
        index=ALL.index
    )


# ----------------------------------------------------------------------
# LanguageTool
# ----------------------------------------------------------------------
def lt_block():

    import language_tool_python as lt

    tool = lt.LanguageTool(
        'en-US'
    )

    g = lambda m, *ns: next(
        (
            getattr(m, n)
            for n in ns
            if getattr(m, n, None)
        ),
        ''
    )

    rows = []

    for f in tqdm(
        ALL.index,
        desc='LanguageTool'
    ):

        # Safe empty transcript handling
        text = TEXT[f] or '.'

        ms = tool.check(
            text
        )

        n = max(
            len(norm_words(TEXT[f])),
            1
        )

        cat = Counter(
            str(
                g(
                    m,
                    'category'
                )
            )
            for m in ms
        )

        rid = [
            str(
                g(
                    m,
                    'ruleId',
                    'rule_id'
                )
            ).upper()
            for m in ms
        ]

        rows.append(
            dict(
                lt_total=len(ms),

                lt_per100=(
                    100 * len(ms) / n
                ),

                lt_grammar=(
                    100 * cat.get(
                        'GRAMMAR',
                        0
                    ) / n
                ),

                lt_typos=(
                    100 * cat.get(
                        'TYPOS',
                        0
                    ) / n
                ),

                lt_punct=(
                    100 * cat.get(
                        'PUNCTUATION',
                        0
                    ) / n
                ),

                lt_casing=(
                    100 * cat.get(
                        'CASING',
                        0
                    ) / n
                ),

                lt_style=(
                    100 * cat.get(
                        'STYLE',
                        0
                    ) / n
                ),

                lt_agree=sum(
                    'AGREEMENT' in r
                    for r in rid
                ),

                lt_article=sum(
                    (
                        ('ARTICLE' in r)
                        or ('A_AN' in r)
                        or ('A_VS_AN' in r)
                    )
                    for r in rid
                ),

                lt_prep=sum(
                    'PREP' in r
                    for r in rid
                ),

                lt_verb=sum(
                    (
                        ('VERB' in r)
                        or ('TENSE' in r)
                    )
                    for r in rid
                )
            )
        )

    return pd.DataFrame(
        rows,
        index=ALL.index
    )


# ----------------------------------------------------------------------
# Run neural grammar detectors
# ----------------------------------------------------------------------
if torch is not None:

    for flag, nm, fn in (
        (CFG.USE_COLA, 'cola', cola_block),
        (CFG.USE_LM, 'lm', lm_block),
        (CFG.USE_GEC, 'gec', gec_block)
    ):

        if flag:

            model_key = {
                'cola': 'COLA_MODEL',
                'lm': 'LM_MODEL',
                'gec': 'GEC_MODEL'
            }[nm]

            r = cached(
                f'{nm}_{tag(getattr(CFG, model_key))}',
                fn
            )

            if r is not None:

                FEATURE_BLOCKS[nm] = r


# ----------------------------------------------------------------------
# LanguageTool
# ----------------------------------------------------------------------
if CFG.USE_LANGUAGETOOL:

    r = cached(
        'languagetool',
        lt_block
    )

    if r is not None:
        FEATURE_BLOCKS['lt'] = r


# ----------------------------------------------------------------------
# Final feature block shapes
# ----------------------------------------------------------------------
print({
    k: v.shape
    for k, v in FEATURE_BLOCKS.items()
})

## 12. Frozen transformer text embeddings

In [ ]:
# ---- 12. Transformer features -----------------------------------------------------------------------------
# Frozen transformer embeddings of the transcript (no fine-tuning: N=769 is too small; PCA is applied inside each CV fold).

def emb_cached(name, fn):
    p = CFG.CACHE / f'{name}.npy'

    if p.exists():
        print(f'Loading cached embeddings: {p}')
        return np.load(p)

    try:
        a = fn()
        np.save(p, a)
        return a

    except Exception as e:
        print(f'[skip {name}]', repr(e)[:300])
        return None


# =============================================================================================================
# SentenceTransformer
# =============================================================================================================

def st_embed():
    from sentence_transformers import SentenceTransformer

    m = SentenceTransformer(
        mpath(CFG.ST_MODEL),
        device=DEVICE
    )

    texts = [
        TEXT[f] if isinstance(TEXT[f], str) and TEXT[f].strip()
        else '.'
        for f in ALL.index
    ]

    return m.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True
    )


# =============================================================================================================
# DeBERTa
# =============================================================================================================

def deberta_embed():
    from transformers import AutoTokenizer, AutoModel

    name = mpath(CFG.DEBERTA_MODEL)

    tok = AutoTokenizer.from_pretrained(name)

    m = AutoModel.from_pretrained(name).to(DEVICE).eval()

    out = []

    texts = [
        TEXT[f] if isinstance(TEXT[f], str) and TEXT[f].strip()
        else '.'
        for f in ALL.index
    ]

    for b in tqdm(
        range(0, len(texts), 16),
        desc='DeBERTa'
    ):
        batch = texts[b:b + 16]

        enc = tok(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512
        ).to(DEVICE)

        with torch.no_grad():
            h = m(**enc).last_hidden_state.float()

        mask = enc.attention_mask.unsqueeze(-1).float()

        pooled = (
            (h * mask).sum(1)
            /
            mask.sum(1).clamp_min(1.0)
        )

        out.append(
            pooled.cpu().numpy()
        )

    return np.vstack(out)


# =============================================================================================================
# Run embeddings
# =============================================================================================================

if CFG.USE_ST_EMB:
    a = emb_cached(
        f'emb_st_{tag(CFG.ST_MODEL)}',
        st_embed
    )

    if a is not None:
        EMB_BLOCKS['embst'] = a


if CFG.USE_DEBERTA_EMB:
    a = emb_cached(
        f'emb_deb_{tag(CFG.DEBERTA_MODEL)}',
        deberta_embed
    )

    if a is not None:
        EMB_BLOCKS['embdeb'] = a


print(
    'Embedding blocks:',
    {
        k: v.shape
        for k, v in EMB_BLOCKS.items()
    }
)

## 13. Whisper-encoder audio embeddings
One more frozen audio view (cached, guarded, GPU only). It enters the feature sets automatically if it was computed.

In [ ]:
# ---- 13. Whisper-encoder audio embeddings ------------------------------------------------------------------------------------------------------
# WHY: the audio-embedding block is the strongest signal in this task. Whisper's encoder was trained on far more varied speech than wav2vec2 and
# is a very strong frozen proficiency representation. Mid / upper-mid layers (mean-pooled over the real frames of each 30 s window) are used.
# Fully guarded: if anything fails the pipeline simply continues without this block.
USE_WHISPER_ENC = HAS_GPU
WENC_MODEL = 'openai/whisper-large-v3' if HAS_GPU else 'openai/whisper-small'
def whisper_encoder_embeddings(paths, max_dur=None):
    from transformers import WhisperFeatureExtractor, WhisperModel as HFWhisperModel
    dt = torch.float16 if HAS_GPU else torch.float32
    fe = WhisperFeatureExtractor.from_pretrained(mpath(WENC_MODEL))
    enc = HFWhisperModel.from_pretrained(mpath(WENC_MODEL), torch_dtype=dt).encoder.to(DEVICE).eval()
    nl = len(enc.layers); layers = (nl // 2, (3 * nl) // 4); E = []
    for p in tqdm(paths, desc='whisper-encoder'):
        y, _ = librosa.load(str(p), sr=16000, mono=True, duration=max_dur)
        chunks = [y[i:i + 30 * 16000] for i in range(0, len(y), 30 * 16000)]            # the encoder sees exactly 30 s windows
        chunks = [c for c in chunks if len(c) >= 3 * 16000] or [y]
        vecs, wts = [], []
        for c in chunks:
            x = fe(c, sampling_rate=16000, return_tensors='pt').input_features.to(DEVICE, dtype=dt)
            with torch.no_grad(): hs = enc(x, output_hidden_states=True).hidden_states
            nfr = max(1, int(len(c) / 16000 * 50))                                       # 50 encoder frames / s; ignore the padded tail
            vecs.append(np.concatenate([hs[l][0, :nfr].float().mean(0).cpu().numpy() for l in layers])); wts.append(nfr)
        E.append(np.average(vecs, axis=0, weights=wts))
    del enc; gc.collect()
    if HAS_GPU: torch.cuda.empty_cache()
    return np.stack(E)

if USE_WHISPER_ENC and torch is not None:
    for _blk, _md in (('embwenc', None),):
        _cp = CFG.CACHE / (f'emb_whisper_enc_{tag(WENC_MODEL)}.npy' if _md is None else f'emb_whisper_enc_{tag(WENC_MODEL)}_d{int(_md)}.npy')
        try:
            E_w = np.load(_cp) if _cp.exists() else whisper_encoder_embeddings(list(ALL.path), max_dur=_md)
            if not _cp.exists(): np.save(_cp, E_w)
            assert E_w.shape[0] == len(ALL) and np.isfinite(E_w).all()
            EMB_BLOCKS[_blk] = E_w; print(f'Whisper-encoder block {_blk}:', E_w.shape)
        except Exception as e:
            print(f'{_blk} skipped:', repr(e)[:200])
else:
    print('Whisper-encoder embeddings disabled (USE_WHISPER_ENC = False).')
print('Embedding blocks:', {k: v.shape for k, v in EMB_BLOCKS.items()})


## 14. Feature matrix
All engineered features + frozen embeddings in one table. Count features become per-minute rates (train clips ~60 s, many test clips ~45 s), extreme values are winsorised on train quantiles, heavy right tails are log-transformed.

In [ ]:
# ---- 14. Feature matrix (label-free cleaning only; imputation / scaling / PCA happen INSIDE each CV fold) ----
GROUP_PREFIX = {'audio': ['aud_'], 'ctc': ['ctc_'], 'asr': ['asr_'], 'txt': ['txt_'], 'syn': ['syn_'],
                'gram': ['gec_', 'cola_', 'lm_', 'lt_']}
F = pd.concat([b for b in FEATURE_BLOCKS.values()], axis=1)
F = F.loc[:, ~F.columns.duplicated()].reindex(ALL.index).replace([np.inf, -np.inf], np.nan)
# Train recordings are ~60 s but a large share of test recordings is ~45 s (see Section 6). Absolute counts/durations would
# shift, so every count-type feature becomes a per-minute RATE and absolute time features are dropped.
_dur_min = (ALL.duration.reindex(F.index) / 60.0).clip(lower=0.1)
COUNT_COLS = [c for c in ['aud_n_segments', 'aud_n_pauses', 'aud_n_pauses_500', 'aud_n_pauses_1s', 'aud_pause_total', 'aud_n_voiced_segs',
              'asr_gaps_300', 'asr_gaps_500', 'asr_gaps_1s', 'asr_n_segments', 'txt_n_words', 'txt_n_sent', 'txt_n_unique', 'txt_n_long',
              'txt_n_repeated', 'txt_fillers', 'txt_um', 'txt_uh', 'txt_like', 'txt_rep_word', 'txt_rep_phrase', 'txt_rep_punct',
              'txt_false_start', 'txt_selfcorr', 'txt_hesitation', 'lt_total', 'lt_agree', 'lt_article', 'lt_prep', 'lt_verb'] if c in F.columns]
for c in COUNT_COLS: F[c + '_pm'] = F[c] / _dur_min
F = F.drop(columns=COUNT_COLS + [c for c in ['aud_dur', 'aud_speech_dur', 'aud_sil_dur', 'aud_lead_sil', 'aud_trail_sil', 'asr_speech_start', 'asr_speech_end',
                                              'asr_word_p_lt50'][:7] if c in F.columns])
print('duration guard: converted', len(COUNT_COLS), 'count features to per-minute rates')
F = F.loc[:, F.loc[TRAIN_NAMES].isna().mean() < 0.4]
F = F.loc[:, F.loc[TRAIN_NAMES].std() > 1e-9]
lo, hi = F.loc[TRAIN_NAMES].quantile(.005), F.loc[TRAIN_NAMES].quantile(.995)     # winsorise using TRAIN quantiles (no labels)
F = F.clip(lo, hi, axis=1)
for c in F.columns:                                                                # tame heavy right tails
    col = F[c]
    if col.min() >= 0 and stats.skew(col.dropna()) > 2: F[c] = np.log1p(col)
GROUPS = {g: [c for c in F.columns if any(c.startswith(p) for p in ps)] for g, ps in GROUP_PREFIX.items()}
GROUPS = {g: c for g, c in GROUPS.items() if c}
EMB_COLS = {}
for b, a in EMB_BLOCKS.items():
    cols = [f'{b}_{i}' for i in range(a.shape[1])]
    F = pd.concat([F, pd.DataFrame(a, index=ALL.index, columns=cols)], axis=1); EMB_COLS[b] = cols
print('groups:', {g: len(c) for g, c in GROUPS.items()}, '| embeddings:', {b: len(c) for b, c in EMB_COLS.items()}, '| total cols:', F.shape[1])
XTR, XTE = F.loc[TRAIN_NAMES], F.loc[TEST_NAMES]
ALLG = list(GROUPS)                       # engineered feature groups used by the model
print('model inputs -> engineered groups:', {g: len(c) for g, c in GROUPS.items()}, '| embedding blocks:', {b: len(c) for b, c in EMB_COLS.items()})

## 15. Validation groups: duplicates + speaker/channel clusters
The nearest neighbour of a recording in an early wav2vec2 layer (which encodes speaker and recording channel) almost always has the same grade, so ordinary random folds leak speaker information. Recordings are clustered on that early-layer embedding and every cluster stays inside one fold; exact duplicates are tied together too. Label-0 recordings (silent/failed takes) are not a speaker and stay single.

In [ ]:
# ---- 15. Validation groups -----------------------------------------------------------------------------------------------------------------------
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import AgglomerativeClustering
names = list(ALL.index); pos = {n: i for i, n in enumerate(names)}; edges = []
d_md5 = ALL.groupby('md5').filename.apply(list); d_md5 = d_md5[d_md5.str.len() > 1]                     # exact duplicate audio
for g in d_md5:
    if len(g) <= 10: edges += [(pos[g[0]], pos[x]) for x in g[1:]]
tr_pos = np.array([pos[n] for n in TRAIN_NAMES]); te_pos = np.array([pos[n] for n in TEST_NAMES])
if SPK_EMB is not None:
    E = SPK_EMB[tr_pos] - SPK_EMB[tr_pos].mean(0); E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
    for k in (CFG.SPK_CLUSTERS, 160, 240, 360):                                                           # no cluster may hold > 10% of the training data
        cl = AgglomerativeClustering(n_clusters=k, metric='cosine', linkage='average').fit_predict(E)
        if np.bincount(cl).max() <= 0.10 * len(tr_pos): break
    first = {}
    for p, c, yy in zip(tr_pos, cl, Y):
        if yy == 0: continue
        if c in first: edges.append((first[c], int(p)))
        else: first[c] = int(p)
    print(f'speaker/channel clusters: k={k} | largest cluster {np.bincount(cl).max()} | median size {np.median(np.bincount(cl)):.0f}')
    # how far is the TEST set from the training data? (same early-layer similarity)
    Et = SPK_EMB[te_pos] - SPK_EMB[tr_pos].mean(0); Et = Et / (np.linalg.norm(Et, axis=1, keepdims=True) + 1e-9)
    S_tt = E @ E.T; np.fill_diagonal(S_tt, -1)
    print(f'median cosine similarity to the nearest TRAIN clip: train->train {np.median(S_tt.max(1)):.3f} | test->train {np.median((Et @ E.T).max(1)):.3f}  '
          '(test speakers/channels are NOT in train -> grouped CV is the right preview, and still a bit generous)')
if edges:
    r_, c_ = zip(*edges); lab = connected_components(csr_matrix((np.ones(len(r_)), (r_, c_)), shape=(len(names),) * 2), directed=False)[1]
else: lab = np.arange(len(names))
GROUPS_TR = lab[tr_pos]; print('CV groups:', len(set(GROUPS_TR)), '| largest group', np.bincount(GROUPS_TR).max())

## 16. Final model: RidgeCV on all features — speaker-grouped CV, training RMSE
One pipeline: median imputation + standardisation of the engineered features; standardisation + PCA (96 components for the two audio embeddings, 32 for the text embeddings) per embedding block; `RidgeCV` (alpha chosen by internal leave-one-out). Predictions are clipped to the rubric range [0, 5].
**Training RMSE** (compulsory) = error on the data the model was fitted on, reported next to the out-of-fold RMSE (honest estimate).

In [ ]:
# ---- 16. Final model, grouped cross-validation, training RMSE --------------------------------------------------------------------------------------
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV

def pearson(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.corrcoef(a, b)[0, 1]) if np.std(a) > 0 and np.std(b) > 0 else np.nan
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

TAB_COLS = [c for g in ALLG for c in GROUPS[g]]
def build_final():
    tl = [('tab', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), TAB_COLS)]
    for b, cols in EMB_COLS.items():
        tl.append((b, Pipeline([('sc', StandardScaler()), ('pca', PCA(n_components=CFG.EMB_PCA_BY_BLOCK.get(b, CFG.EMB_PCA), random_state=CFG.SEED))]), cols))
    return Pipeline([('prep', ColumnTransformer(tl, remainder='drop')), ('ridge', RidgeCV(alphas=np.logspace(-1, 4, 30)))])

def make_splits(y, groups, n_splits, n_repeats, seed):
    ybin = np.digitize(y, np.quantile(y, [.2, .4, .6, .8])); out = []
    for r in range(n_repeats):
        for k, (a, b) in enumerate(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed + r).split(np.zeros(len(y)), ybin, groups)): out.append((r, k, a, b))
    return out
SPLITS = make_splits(Y, GROUPS_TR, CFG.N_SPLITS, CFG.N_REPEATS, CFG.SEED)

# ---- speaker-grouped repeated CV ------------------------------------------------------------------------------------------------------------------
oof = np.full((CFG.N_REPEATS, len(Y)), np.nan); rows = []
for r, k, a, b in SPLITS:
    m = build_final().fit(XTR.iloc[a], Y[a]); pv = np.clip(m.predict(XTR.iloc[b]), 0, 5); pt = np.clip(m.predict(XTR.iloc[a]), 0, 5); oof[r, b] = pv
    rows.append(dict(repeat=r, fold=k, train_RMSE=rmse(Y[a], pt), val_RMSE=rmse(Y[b], pv), val_Pearson=pearson(Y[b], pv)))
FOLDS = pd.DataFrame(rows); OOF = np.nanmean(oof, 0); nz = Y > 0
# ---- final model fitted on ALL training data -> training RMSE ---------------------------------------------------------------------------------------
FINAL = build_final().fit(XTR, Y); train_pred = np.clip(FINAL.predict(XTR), 0, 5)
TRAIN_RMSE, TRAIN_PEARSON = rmse(Y, train_pred), pearson(Y, train_pred)
print('=' * 74)
print(f'TRAINING RMSE (final model, in-sample, n={len(Y)})  : {TRAIN_RMSE:.4f}   | Pearson {TRAIN_PEARSON:.4f}')
print(f'Speaker-grouped CV, mean over {len(FOLDS)} folds     : RMSE {FOLDS.val_RMSE.mean():.4f} ± {FOLDS.val_RMSE.std():.4f}   | Pearson {FOLDS.val_Pearson.mean():.4f} ± {FOLDS.val_Pearson.std():.4f}')
print(f'Out-of-fold (repeat-averaged), all rows            : RMSE {rmse(Y, OOF):.4f}   | Pearson {pearson(Y, OOF):.4f}')
print(f'Out-of-fold, rows with label > 0                   : RMSE {rmse(Y[nz], OOF[nz]):.4f}   | Pearson {pearson(Y[nz], OOF[nz]):.4f}')
print(f'Baseline: predict the mean (= label SD)            : RMSE {Y.std():.4f}')
print(f'Chosen ridge alpha (full fit): {FINAL.named_steps["ridge"].alpha_:.2f}')
print('=' * 74)

# ---- plots ----------------------------------------------------------------------------------------------------------------------------------------
rng = np.random.RandomState(0); fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ax[0].scatter(Y + rng.uniform(-.08, .08, len(Y)), OOF, s=9, alpha=.5); ax[0].plot([0, 5], [0, 5], 'r--', lw=1)
ax[0].set(xlabel='true grammar score', ylabel='predicted (out-of-fold)', title=f'Out-of-fold: RMSE {rmse(Y, OOF):.3f} | r {pearson(Y, OOF):.3f}')
res = pd.DataFrame({'true': Y, 'resid': OOF - Y}); res.boxplot(column='resid', by='true', ax=ax[1], grid=False); ax[1].axhline(0, color='r', lw=1)
ax[1].set(title='Residual by true score (regression to the mean at the extremes)', xlabel='true score', ylabel='prediction - truth'); plt.sca(ax[1]); plt.xticks(rotation=0)
ax[2].bar(['train (in-sample)', 'grouped CV', 'mean baseline'], [TRAIN_RMSE, FOLDS.val_RMSE.mean(), Y.std()], color=['seagreen', 'tomato', 'grey']); ax[2].set_title('RMSE'); ax[2].set_ylabel('RMSE')
plt.suptitle(''); plt.tight_layout(); plt.show()

## 17. Interpretation: which inputs drive the predictions?
Standardised ridge coefficients of the engineered features, and the share of prediction variance contributed by each feature group / embedding block.

In [ ]:
# ---- 17. Interpretation ---------------------------------------------------------------------------------------------------------------------------
prep, ridge = FINAL.named_steps['prep'], FINAL.named_steps['ridge']
Z = prep.transform(XTR); coef = ridge.coef_; n_tab = len(TAB_COLS)
assert Z.shape[1] == n_tab + sum(prep.named_transformers_[b].named_steps['pca'].n_components_ for b in EMB_COLS), 'unexpected column layout'
cs = pd.Series(coef[:n_tab], index=TAB_COLS)
top = cs.reindex(cs.abs().sort_values(ascending=False).index[:20])[::-1]
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
top.plot.barh(ax=ax[0], color=['seagreen' if v > 0 else 'tomato' for v in top]); ax[0].set_title('Top-20 engineered features (standardised ridge coefficient)')
contrib = {}
for g in ALLG: idx = [TAB_COLS.index(c) for c in GROUPS[g]]; contrib[g] = float(np.std(Z[:, idx] @ coef[idx]))
off = n_tab
for b in EMB_COLS: nc = prep.named_transformers_[b].named_steps['pca'].n_components_; contrib[b] = float(np.std(Z[:, off:off + nc] @ coef[off:off + nc])); off += nc
ct = pd.Series(contrib).sort_values(); ct.plot.barh(ax=ax[1], color='steelblue'); ax[1].set_title('Std. of the prediction contributed by each input block'); plt.tight_layout(); plt.show()
print('Block contribution (std of prediction):', ct.round(3).sort_values(ascending=False).to_dict())

## 18. Test predictions and submission file
The fitted final model predicts the 216 test recordings. The file follows `sample_submission.csv` exactly (`filename,label`, ids copied verbatim, label in [0, 5]).

In [ ]:
# ---- 18. Test prediction + submission ---------------------------------------------------------------------------------------------------------------
test_pred = np.clip(FINAL.predict(XTE), CFG.LABEL_BOUNDS[0], CFG.LABEL_BOUNDS[1])
print(f'test predictions: n={len(test_pred)} | mean {test_pred.mean():.3f} | sd {test_pred.std():.3f} | min {test_pred.min():.2f} | max {test_pred.max():.2f}   (train labels: mean {Y.mean():.3f}, sd {Y.std():.3f})')
plt.figure(figsize=(7, 3.6)); plt.hist(Y, bins=np.arange(-.25, 5.75, .5), alpha=.55, density=True, label='train labels'); plt.hist(test_pred, bins=np.arange(-.25, 5.75, .5), alpha=.55, density=True, label='test predictions')
plt.xlabel('grammar score'); plt.legend(); plt.title('Label vs prediction distribution'); plt.tight_layout(); plt.show()

pred_map = {n.split('/', 1)[1]: float(v) for n, v in zip(TEST_NAMES, test_pred)}          # key = raw name without extension
def fill_template(template):
    out = template[[ID_COL]].copy(); out[TARGET_COL] = out[ID_COL].map(strip_ext).map(pred_map); return out
sub_sample, sub_testcsv = fill_template(sample_sub), fill_template(test_raw)
cov_sample, cov_testcsv = sub_sample[TARGET_COL].notna().mean(), sub_testcsv[TARGET_COL].notna().mean()
print(f'coverage | sample_submission ids: {cov_sample:.1%} ({sub_sample[TARGET_COL].notna().sum()}/{len(sub_sample)}) | test.csv ids: {cov_testcsv:.1%}')
def finalize(df, name):
    n_bad = int(df[TARGET_COL].isna().sum())
    if n_bad: print(f'  [{name}] WARNING: {n_bad} rows have no audio -> filled with the train mean {Y.mean():.3f} (no information in those rows).')
    df = df.copy(); df[TARGET_COL] = df[TARGET_COL].fillna(Y.mean()).clip(0, 5).round(4)
    assert list(df.columns) == list(sample_sub.columns) and df[ID_COL].is_unique and np.isfinite(df[TARGET_COL]).all()
    return df
finalize(sub_sample, 'sample ids').to_csv(CFG.WORK / 'submission_sample_ids.csv', index=False)
finalize(sub_testcsv, 'test.csv ids').to_csv(CFG.WORK / 'submission_testcsv_ids.csv', index=False)
primary = sub_sample if cov_sample >= 0.99 else sub_testcsv
finalize(primary, 'submission.csv').to_csv(CFG.WORK / 'submission.csv', index=False)
final_sub = pd.read_csv(CFG.WORK / 'submission.csv')
print('\nsubmission.csv is built from', 'the sample_submission ids' if cov_sample >= 0.99 else 'the test.csv ids (the sample_submission ids are not predictable from the supplied audio)')
print(final_sub.head(), '\nshape:', final_sub.shape, '| location:', CFG.WORK / 'submission.csv')

## 19. Report
**Approach.** Speech → text (Whisper large-v3 with a disfluency-preserving prompt, plus a literal wav2vec2-CTC transcript) → linguistic features; acoustic features; frozen audio and text embeddings → one ridge regression.

**Preprocessing.** 16 kHz mono audio, validated (readability, sample rate, duration, md5 duplicates). Count features converted to per-minute rates (train clips ~60 s vs ~45 s for many test clips); winsorisation with train quantiles; log transform of heavy right tails; imputation, scaling and PCA fitted inside each CV fold only.

**Features.**
- *Acoustic:* pauses, speaking/articulation rate, MFCC, spectral and F0 statistics.
- *ASR:* word probabilities, timing, Whisper-vs-literal disagreement, CTC confidence.
- *Text:* disfluencies, lexical and sentence-structure features, spaCy syntax, CoLA acceptability, GPT-2 surprisal, T5 grammar-correction edit distance, LanguageTool counts.
- *Embeddings:* wav2vec2 and Whisper-encoder (audio), sentence-transformer and DeBERTa (text).

**Model.** RidgeCV on the concatenation of the scaled engineered features and per-block PCA embeddings; output clipped to [0, 5]. In speaker-grouped CV linear models on these embeddings were more accurate and more stable than SVR, gradient boosting, random forests and stacked ensembles, so a single regularised linear model is used.

**Evaluation.** Training RMSE and out-of-fold RMSE / Pearson are printed in Section 16, with scatter, residual and RMSE plots. Validation is speaker-grouped (clusters of similar speaker/channel embeddings never straddle a fold) because test recordings are far less similar to the training data than training recordings are to each other; ordinary random folds overstate accuracy (≈0.50 vs ≈0.74 RMSE for an SVR in my earlier comparison).

**Interpretation.** Section 17 shows that the audio embeddings carry most of the signal, with ASR confidence, disfluency and lexical-diversity features adding complementary information.

**Limitations.** Small N (769); the test set contains speakers/channels unseen in training, so the leaderboard score is expected to be close to, or somewhat worse than, the grouped-CV numbers; predictions regress toward the mean at the extremes of the scale.